# Managing Authentication & Authorization

This notebook shows how authentication and authorization work in istSOS4.

In simple terms:

- **Authentication** answers the question: *who are you?*  
  This happens when a user logs in and receives an access token.
- **Authorization** answers the question: *what are you allowed to do?*  
  This is controlled through roles and policies.

A **role** describes the type of user, application, or device.  
A **policy** gives that role the actual permissions to access or manage data.

> Creating a user is not enough by itself.  
> The user also needs a policy, otherwise the system does not know what that user is allowed to do.

## Roles and permissions in istSOS4

The table below summarizes the main roles used in istSOS4.

| Role | Who or what uses it | What it can do | Typical use |
|---|---|---|---|
| `admin` | System administrator | Has full access to the system. The administrator can manage data and resources, and is also the only role that can create users and policies. | Full system management, initial setup, access management, user management |
| `viewer` | People or applications that only need to consult data | Reads data, but cannot create, change, or delete it. | Dashboards, public applications, reporting tools |
| `editor` | Authorized users or services that manage system resources | Reads, creates, updates, and deletes data according to the assigned policy. | System configuration, maintenance, data management |
| `sensor` | Devices, sensors, data loggers, or automatic ingestion services | Sends new observations and can update some operational information, such as location or datastream information. | IoT devices, automatic data collection, data ingestion |
| `obs_manager` | Users or services responsible for observation management | Manages observations more extensively, including correcting or deleting existing observations when needed. | Data validation, quality control, correction of wrong observations |

## Preliminary steps

First, we import the required Python libraries and define the base URL of the istSOS4 API.

When this notebook runs inside the Jupyter Docker container, the API should be reached through the Docker service name:

```python
IST_SOS_ENDPOINT = "http://api:5000/v4/v1.1"
```

If you run the same code directly from your host machine instead of inside Docker, you may need to use the external port exposed by Docker Compose.


In [1]:
from datetime import datetime
import json

import requests
from IPython.display import Markdown, display

from istsos_utils import (
    REQUEST_TIMEOUT,
    auth_headers,
    display_error_response,
    display_json,
    login,
)

IST_SOS_ENDPOINT = "http://api:5000/v4/v1.1"

## Step 1 — Login as administrator

We start by logging in as the administrator.

The administrator account is special because it has full access to the system. In this notebook, we mainly use it for access-management tasks:

- creating new users;
- assigning a role to each user;
- creating policies that define what each user is allowed to do.

The administrator can also perform the other operations shown in the notebook, such as reading and creating resources. However, here we use it mainly to prepare the `viewer` and `editor` users used in the examples.

The login request returns an access token. This token is then sent with the following requests to prove that we are authenticated as the administrator.


In [2]:
admin_username = input("Enter administrator username: ")
admin_password = input("Enter administrator password: ")

if not admin_username or not admin_password:
    print("Username or password is empty")
else:
    token, login_body = login(
        IST_SOS_ENDPOINT,
        admin_username,
        admin_password,
        timeout=REQUEST_TIMEOUT,
    )

    if token:
        print("Logged in as administrator")

Enter administrator username:  admin
Enter administrator password:  admin


Logged in as administrator


## Step 2 — Create a `viewer` user

Now the administrator creates a new user with the `viewer` role.

The role describes the type of user we are creating. In this case, the user is intended to read data only.

> This request must be executed with an administrator token. A normal user cannot create other users.


In [3]:
viewer_username = "group1_viewer"
viewer_password = "qwertz"

user = {
    "username": viewer_username,
    "password": viewer_password,
    "role": "viewer",
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Users",
    headers=auth_headers(token),
    json=user,
    timeout=REQUEST_TIMEOUT,
)

if response.status_code in (200, 201):
    print("Viewer user created successfully")
else:
    display_error_response(response)


Viewer user created successfully


## Step 3 — Create a policy for the `viewer`

Creating the user is not enough by itself. The user also needs a policy.

A policy is the rule that gives the user actual permissions on the data. For a `viewer`, the policy allows reading data but does not allow creating, updating, or deleting it.

> This request must also be executed by the administrator. Only the administrator can create or assign policies.


In [4]:
policy = {
    "users": [viewer_username],
    "name": "workshop",
    "permissions": {"type": "viewer"},
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Policies",
    headers=auth_headers(token),
    json=policy,
    timeout=REQUEST_TIMEOUT,
)

if response.status_code in (200, 201):
    print("Viewer policy created successfully")
else:
    display_error_response(response)

Viewer policy created successfully


## Step 4 — Login as the `viewer`

We now switch from the administrator account to the newly created `viewer` user.

From this point on, the token belongs to the `viewer`, so the following requests will be executed with viewer permissions.


In [5]:
username = input("Enter viewer username: ")
password = input("Enter viewer password: ")

if not username or not password:
    print("Username or password is empty")
else:
    token, login_body = login(
        IST_SOS_ENDPOINT,
        username,
        password,
        timeout=REQUEST_TIMEOUT,
    )

    if token:
        prefix = username + "_"
        print("Logged in as viewer")

Enter viewer username:  group1_viewer
Enter viewer password:  qwertz


Logged in as viewer


## Step 5 — Test the `viewer` permissions

We will now test what the `viewer` can and cannot do.

The expected behaviour is:

- reading data should work;
- creating new data should fail.


### Retrieve data as `viewer`

The `viewer` is allowed to read data, so this request should succeed.


In [6]:
response = requests.get(
    IST_SOS_ENDPOINT + "/Things",
    headers=auth_headers(token),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 200:
    display_json(response.json())
else:
    display_error_response(response)

```json
{
  "value": []
}
```

### Try to create data as `viewer`

The `viewer` is not allowed to create new data.

This request is intentionally included to show that authorization is working. The expected result is an error response, because the `viewer` can read data but cannot add new resources.


#### Create a `Thing`

A `Thing` represents an entity being observed or monitored. In this case, we try to create an example `Thing` for Lugano Lake.

Because we are still logged in as `viewer`, this operation should not be allowed.


In [7]:
thing = {
    "name": "Lugano Lake",
    "description": "The Alpine lake located in Southern Switzerland",
    "properties": {
        "Max depth": "288 m",
    },
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Things",
    json=thing,
    headers=auth_headers(token, "Create new thing"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    print(f"Thing created successfully ({response.headers['location']})")
else:
    print("The Thing was not created.")
    display_error_response(response)

The Thing was not created.


```json
{
  "code": 403,
  "type": "error",
  "message": "Insufficient privileges."
}
```

## Step 6 — Login again as administrator

To create another user, we need to switch back to the administrator account.

This is necessary because user and policy management is reserved for the administrator.


In [8]:
if not admin_username or not admin_password:
    print("Administrator username or password is empty")
else:
    token, login_body = login(
        IST_SOS_ENDPOINT,
        admin_username,
        admin_password,
        timeout=REQUEST_TIMEOUT,
    )

    if token:
        print("Logged in as administrator")

Logged in as administrator


## Step 7 — Create an `editor` user

The administrator now creates a new user with the `editor` role.

An `editor` is allowed to manage data, not only read it. This means that the user can create, modify, and delete resources according to the assigned policy.


In [9]:
editor_username = "group1_editor"
editor_password = "qwertz"

user = {
    "username": editor_username,
    "password": editor_password,
    "role": "editor",
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Users",
    headers=auth_headers(token),
    json=user,
    timeout=REQUEST_TIMEOUT,
)

if response.status_code in (200, 201):
    print("Editor user created successfully")
else:
    display_error_response(response)

Editor user created successfully


## Step 8 — Create a policy for the `editor`

As before, the user also needs a policy.

The `editor` policy gives broader permissions than the `viewer` policy. It allows the user to manage the main resources in istSOS4.

> This step is performed by the administrator because policies define access rules for the system.


In [10]:
policy = {
    "users": [editor_username],
    "name": "workshop",
    "permissions": {"type": "editor"},
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Policies",
    headers=auth_headers(token),
    json=policy,
    timeout=REQUEST_TIMEOUT,
)

if response.status_code in (200, 201):
    print("Editor policy created successfully")
else:
    display_error_response(response)

Editor policy created successfully


## Step 9 — Login as the `editor`

We now log in as the `editor` user.

The next requests will be executed with editor permissions, so we expect both reading and creating data to work.


In [11]:
username = input("Enter editor username: ")
password = input("Enter editor password: ")

if not username or not password:
    print("Username or password is empty")
else:
    token, login_body = login(
        IST_SOS_ENDPOINT,
        username,
        password,
        timeout=REQUEST_TIMEOUT,
    )

    if token:
        prefix = username + "-"
        print("Logged in as editor")
        print("Your station name will be prefixed with: " + prefix)

Enter editor username:  group1_editor
Enter editor password:  qwertz


Logged in as editor
Your station name will be prefixed with: group1_editor-


## Step 10 — Test the `editor` permissions

We will now test the permissions assigned to the `editor`.

The expected behaviour is:

- reading data should work;
- creating new data should also work.


### Create data as `editor`

Unlike the `viewer`, the `editor` is allowed to create new resources.

This request should succeed and create a new `Thing`.


#### Create a `Thing`

Here we create the same example resource again, but this time using the `editor` token.

Because the `editor` has write permissions, this operation should be accepted by the API.


In [12]:
thing = {
    "name": f"{prefix}Lugano Lake",
    "description": "The Alpine lake located in Southern Switzerland",
    "properties": {
        "Max depth": "288 m",
    },
}

response = requests.post(
    IST_SOS_ENDPOINT + "/Things",
    json=thing,
    headers=auth_headers(token, "Create new thing"),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 201:
    print(f"Thing created successfully ({response.headers['location']})")
else:
    display_error_response(response)

Thing created successfully (http://localhost:8018/v4/v1.1/Things(2))


### Retrieve data as `editor`

The `editor` is allowed to read data, so this request should succeed.


In [13]:
response = requests.get(
    IST_SOS_ENDPOINT + "/Things",
    headers=auth_headers(token),
    timeout=REQUEST_TIMEOUT,
)

if response.status_code == 200:
    display_json(response.json())
else:
    display_error_response(response)

```json
{
  "@iot.as_of": "2026-05-20T16:07:51Z",
  "value": [
    {
      "@iot.id": 2,
      "@iot.selfLink": "http://localhost:8018/v4/v1.1/Things(2)",
      "Locations@iot.navigationLink": "http://localhost:8018/v4/v1.1/Things(2)/Locations",
      "HistoricalLocations@iot.navigationLink": "http://localhost:8018/v4/v1.1/Things(2)/HistoricalLocations",
      "Datastreams@iot.navigationLink": "http://localhost:8018/v4/v1.1/Things(2)/Datastreams",
      "Commit@iot.navigationLink": "http://localhost:8018/v4/v1.1/Things(2)/Commit(2)",
      "name": "group1_editor-Lugano Lake",
      "description": "The Alpine lake located in Southern Switzerland",
      "properties": {
        "Max depth": "288 m"
      }
    }
  ]
}
```